# Sector Summaries for All Periods

This notebook generates:
1. **Sector Imputation Summary**: Shows the percentage of stocks with imputed scope emissions by sector
2. **Sector Emissions Summary**: Shows emission statistics by sector

Both summaries are created for each period and then aggregated across all periods.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

## Configuration

In [2]:
# Define periods to analyze
PERIODS = ["0321", "0621", "0921", "1221", "0322", "0622", "0922", "1222", "0323", "0623", "0923", "1223"]

# Data directory
DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
DATASETS_DIR = DATA_DIR / "datasets"
OPTIMAL_PORT_DIR = RESULTS_DIR / "optimal_portfolios"
# Display options
pd.options.display.float_format = '{:,.2f}'.format
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

## Helper Functions

In [3]:
def create_sector_imputation_summary(df, period):
    """
    Create sector imputation summary for a single period.
    Shows the percentage of stocks with imputed scope emissions by sector.
    
    Note: Excludes manual imputation columns (only uses automatic imputation flags)
    """
    auto_impute_flags = ['Scope 1 Imputed', 'Scope 2 Imputed', 'Scope 3 Imputed']
    filled_count_cols = ['Filled Scope 1 Count', 'Filled Scope 2 Count', 'Filled Scope 3 Count']
    
    summary_rows = []
    
    for sector, group in df.groupby('GICS Sector'):
        n_stocks = len(group)
        row = {
            'Period': period,
            'Sector': sector,
            'Number of Stocks': n_stocks
        }
        
        # Automatically imputed (Scope 1/2/3)
        for col in auto_impute_flags:
            count = group[col].fillna(0).sum()  # Treat NaN as 0 (not imputed)
            scope = col.split()[1]  # e.g. "1" from "Scope 1 Imputed"
            row[f'Imputed Scope {scope} Count'] = int(count)
            row[f'Imputed Scope {scope} %'] = count / n_stocks * 100
        
        # Add filled counts - now summing binary indicators (0 or 1) per sector
        for col in filled_count_cols:
            if col in group.columns:
                # Sum the binary indicators to get count of filled companies in this sector
                filled_count = group[col].fillna(0).sum()
                scope = col.split()[2]  # e.g. "1" from "Filled Scope 1 Count"
                row[f'Filled Scope {scope} Count'] = int(filled_count)
                row[f'Filled Scope {scope} %'] = (filled_count / n_stocks * 100) if n_stocks > 0 else 0
        
        summary_rows.append(row)
    
    summary_df = pd.DataFrame(summary_rows)
    summary_df = summary_df.sort_values("Sector").reset_index(drop=True)
    
    # Round all percentage columns to integers
    percentage_cols = [col for col in summary_df.columns if '%' in col]
    summary_df[percentage_cols] = summary_df[percentage_cols].round(0).astype('Int64')
    
    # Create 'Missing Scope' = max imputed across scopes
    summary_df["Missing Scope Count"] = summary_df[
        ["Imputed Scope 1 Count", "Imputed Scope 2 Count", "Imputed Scope 3 Count"]
    ].max(axis=1)
    
    summary_df["Missing Scope (%)"] = (
        summary_df["Missing Scope Count"] / summary_df["Number of Stocks"] * 100
    ).round(0).astype('Int64')
    
    # Define column order - include filled counts and percentages
    columns_order = ['Period', 'Sector', 'Number of Stocks', 'Missing Scope Count', 'Missing Scope (%)'] + \
                    ['Filled Scope 1 Count', 'Filled Scope 1 %', 
                     'Filled Scope 2 Count', 'Filled Scope 2 %',
                     'Filled Scope 3 Count', 'Filled Scope 3 %'] + \
                    [col for col in summary_df.columns if col not in 
                     ['Period', 'Sector', 'Number of Stocks', 'Missing Scope Count', 'Missing Scope (%)',
                      'Filled Scope 1 Count', 'Filled Scope 1 %',
                      'Filled Scope 2 Count', 'Filled Scope 2 %',
                      'Filled Scope 3 Count', 'Filled Scope 3 %']]
    
    # Only include columns that exist
    columns_order = [col for col in columns_order if col in summary_df.columns]
    
    summary_df = summary_df[columns_order].sort_values("Missing Scope (%)", ascending=False).reset_index(drop=True)
    
    return summary_df

In [4]:
def sector_aggregates(df, period, top_weight_pct=0.2):
    """
    Create sector emissions summary for a single period.
    Shows emission statistics by sector.
    """
    results = []
    
    total_mcap = df["float_mcap"].sum()  # total benchmark market cap
    
    for sector, g in df.groupby("GICS Sector"):
        # Calculate Scope 1+2+3 if not present
        if 'Scope 1+2+3' not in g.columns:
            g['Scope 1+2+3'] = g['Scope 1'] + g['Scope 2'] + g['Scope 3']
        
        # absolute totals
        s1 = g["Scope 1"].sum()
        s2 = g["Scope 2"].sum()
        s3 = g["Scope 3"].sum()
        total_emissions = g["Scope 1+2+3"].sum()
        
        # percentages of each scope
        pct_s1 = s1/total_emissions if total_emissions > 0 else 0
        pct_s2 = s2/total_emissions if total_emissions > 0 else 0
        pct_s3 = s3/total_emissions if total_emissions > 0 else 0
        
        # top 5 emitters in sector
        top5 = g.nlargest(5, "Scope 1+2+3")["Scope 1+2+3"].sum() / total_emissions if total_emissions > 0 else 0
        
        # emissions captured within top X% of sector weights
        g_sorted = g.sort_values("weight_in_sector", ascending=False).copy()
        g_sorted["cum_weight"] = g_sorted["weight_in_sector"].cumsum()
        emissions_top_weight = g_sorted.loc[g_sorted["cum_weight"] <= top_weight_pct, "Scope 1+2+3"].sum()
        pct_emissions_top_weight = emissions_top_weight / total_emissions if total_emissions > 0 else 0
        
        # sector market cap as percentage of benchmark
        sector_mcap = g["float_mcap"].sum()
        sector_mcap_pct = sector_mcap / total_mcap if total_mcap > 0 else 0
        
        results.append({
            "Period": period,
            "Sector": sector,
            "Scope 1 sum": s1,
            "Scope 2 sum": s2,
            "Scope 3 sum": s3,
            "Scope 1+2+3 absolute": total_emissions,
            "% Scope 1": pct_s1,
            "% Scope 2": pct_s2,
            "% Scope 3": pct_s3,
            "% Top 5 emitters": top5,
            f"% Emissions in Top {int(top_weight_pct*100)}% Weight": pct_emissions_top_weight,
            "Avg Scope 1+2+3": g["Scope 1+2+3"].mean(),
            "Avg Scope 1": g["Scope 1"].mean(),
            "Avg Scope 2": g["Scope 2"].mean(),
            "Avg Scope 3": g["Scope 3"].mean(),
            "Var Scope 1": g["Scope 1"].std(),
            "Var Scope 2": g["Scope 2"].std(),
            "Var Scope 3": g["Scope 3"].std(),
            "Avg Carbon Intensity": g["Carbon Intensity"].mean(),
            "Aggregate Carbon Intensity": total_emissions / g["Revenue"].sum() if g["Revenue"].sum() > 0 else 0,
            "Sector Market Cap %": sector_mcap_pct
        })
    
    return pd.DataFrame(results)

In [5]:
def human_format(num):
    """Format large numbers with K, M, B, T suffixes"""
    for unit in ['', 'K', 'M', 'B', 'T']:
        if abs(num) < 1000:
            return f"{num:.1f}{unit}"
        num /= 1000
    return f"{num:.1f}P"

## Load Data for All Periods

In [6]:
import pickle

In [7]:
all_optimal_portfolios = {}

for period in PERIODS:
    file_path = OPTIMAL_PORT_DIR / f"optimal_portfolios_all_te_{period}.pkl"
    
    if file_path.exists():
        with open(file_path, "rb") as f:
            all_optimal_portfolios[period] = pickle.load(f)
    else:
        print(f"⚠️ Missing optimal portfolio file for period {period}")


    

In [8]:
optimal_portfolios_2_pct_across_time_periods = {}
from utils import extract_optimal_portfolios_at_target_te
for period in all_optimal_portfolios:
    optimal_portfolios_2_pct_across_time_periods[period] = extract_optimal_portfolios_at_target_te(all_optimal_portfolios[period], target_te_bps=200)

In [9]:
def compute_hhi(w):
    w = np.asarray(w, dtype=float)
    return float(np.sum(w**2))

records = []

for period, sectors in all_optimal_portfolios.items():
    for sector, data in sectors.items():
        w_opt = data.get("weights_by_te", None)
        if w_opt is None:
            continue
        
        hhi = compute_hhi(w_opt)
        
        records.append({
            "Period": period,
            "Sector": sector,
            "HHI": hhi
        })

df_hhi = pd.DataFrame(records)

avg_hhi = (
    df_hhi.groupby("Sector")["HHI"]
          .mean()
          .reset_index()
          .rename(columns={"HHI": "Avg_HHI_optimized"})
)

print(avg_hhi)


                    Sector  Avg_HHI_optimized
0   Communication Services              22.42
1   Consumer Discretionary              14.23
2         Consumer Staples               6.89
3                   Energy              17.03
4               Financials               5.09
5              Health Care               5.26
6              Industrials               4.94
7   Information Technology              14.15
8                Materials               7.32
9              Real Estate               7.30
10               Utilities              12.03


In [10]:
# Load benchmark data for all periods
all_periods_data = {}

for period in PERIODS:
    file_path = DATASETS_DIR / f"benchmark_weights_carbon_intensity_{period}.xlsx"
    
    if file_path.exists():
        df = pd.read_excel(file_path)

        # Calculate Scope 1+2+3 if not present
        if 'Scope 1+2+3' not in df.columns:
            df['Scope 1+2+3'] = df['Scope 1'] + df['Scope 2'] + df['Scope 3']
        
        # Calculate float_mcap from weight_in_sector if not present
        if 'float_mcap' not in df.columns:
            # Approximate float_mcap from weights (will be normalized by sector later)
            df['float_mcap'] = df['weight_in_sector']
        
        all_periods_data[period] = df
        print(f"Loaded {period}: {len(df)} companies, {df['GICS Sector'].nunique()} sectors")
    else:
        print(f"Warning: File not found for period {period}")

print(f"\nSuccessfully loaded data for {len(all_periods_data)} periods")

Loaded 0321: 497 companies, 11 sectors
Loaded 0621: 497 companies, 11 sectors
Loaded 0921: 497 companies, 11 sectors
Loaded 1221: 497 companies, 11 sectors
Loaded 0322: 498 companies, 11 sectors
Loaded 0622: 498 companies, 11 sectors
Loaded 0922: 498 companies, 11 sectors
Loaded 1222: 498 companies, 11 sectors
Loaded 0323: 497 companies, 11 sectors
Loaded 0623: 498 companies, 11 sectors
Loaded 0923: 497 companies, 11 sectors
Loaded 1223: 496 companies, 11 sectors

Successfully loaded data for 12 periods


## 1. Sector Imputation Summary

### Individual Period Summaries

In [11]:
# Create imputation summary for each period
imputation_summaries = []

for period, df in all_periods_data.items():
    summary = create_sector_imputation_summary(df, period)
    imputation_summaries.append(summary)
    
    print(f"\n{'='*80}")
    print(f"Imputation Summary - Period {period}")
    print(f"{'='*80}")
    display(summary)


Imputation Summary - Period 0321


,Period,Sector,Number of Stocks,Missing Scope Count,Missing Scope (%),Filled Scope 1 Count,Filled Scope 1 %,Filled Scope 2 Count,Filled Scope 2 %,Filled Scope 3 Count,Filled Scope 3 %,Imputed Scope 1 Count,Imputed Scope 1 %,Imputed Scope 2 Count,Imputed Scope 2 %,Imputed Scope 3 Count,Imputed Scope 3 %
0,0321,Energy,23,11,48,0,0,0,0,2,9,0,0,0,0,11,48
1,0321,Industrials,74,22,30,3,4,3,4,5,7,4,5,4,5,22,30
2,0321,Health Care,63,17,27,4,6,4,6,12,19,4,6,4,6,17,27
3,0321,Financials,74,15,20,4,5,4,5,5,7,4,5,4,5,15,20
4,0321,Consumer Discretionary,58,11,19,1,2,2,3,9,16,2,3,3,5,11,19
5,0321,Information Technology,63,11,17,3,5,3,5,5,8,5,8,5,8,11,17
6,0321,Real Estate,29,5,17,1,3,1,3,4,14,1,3,1,3,5,17
7,0321,Communication Services,22,3,14,2,9,2,9,5,23,3,14,3,14,3,14
8,0321,Consumer Staples,36,4,11,1,3,1,3,0,0,0,0,0,0,4,11
9,0321,Materials,27,2,7,0,0,0,0,5,19,0,0,0,0,2,7



Imputation Summary - Period 0621


,Period,Sector,Number of Stocks,Missing Scope Count,Missing Scope (%),Filled Scope 1 Count,Filled Scope 1 %,Filled Scope 2 Count,Filled Scope 2 %,Filled Scope 3 Count,Filled Scope 3 %,Imputed Scope 1 Count,Imputed Scope 1 %,Imputed Scope 2 Count,Imputed Scope 2 %,Imputed Scope 3 Count,Imputed Scope 3 %
0,0621,Energy,22,10,45,0,0,0,0,2,9,0,0,0,0,10,45
1,0621,Industrials,74,22,30,3,4,3,4,5,7,4,5,4,5,22,30
2,0621,Health Care,63,17,27,4,6,4,6,12,19,4,6,4,6,17,27
3,0621,Financials,74,15,20,4,5,4,5,5,7,4,5,4,5,15,20
4,0621,Consumer Discretionary,58,11,19,1,2,2,3,8,14,2,3,3,5,11,19
5,0621,Information Technology,63,11,17,3,5,3,5,4,6,5,8,5,8,11,17
6,0621,Real Estate,29,5,17,1,3,1,3,4,14,1,3,1,3,5,17
7,0621,Communication Services,22,3,14,1,5,1,5,5,23,3,14,3,14,3,14
8,0621,Consumer Staples,36,4,11,1,3,1,3,1,3,0,0,0,0,4,11
9,0621,Materials,28,2,7,0,0,0,0,5,18,0,0,0,0,2,7



Imputation Summary - Period 0921


...

,Period,Sector,Number of Stocks,Missing Scope Count,Missing Scope (%),Filled Scope 1 Count,Filled Scope 1 %,Filled Scope 2 Count,Filled Scope 2 %,Filled Scope 3 Count,Filled Scope 3 %,Imputed Scope 1 Count,Imputed Scope 1 %,Imputed Scope 2 Count,Imputed Scope 2 %,Imputed Scope 3 Count,Imputed Scope 3 %
0,0623,Energy,23,11,48,2,9,2,9,2,9,2,9,2,9,11,48
1,0623,Health Care,64,20,31,1,2,1,2,4,6,7,11,7,11,20,31
2,0623,Industrials,75,22,29,2,3,2,3,3,4,5,7,5,7,22,29
3,0623,Real Estate,31,8,26,0,0,0,0,0,0,4,13,4,13,8,26
4,0623,Financials,72,16,22,2,3,2,3,3,4,6,8,6,8,16,22
5,0623,Information Technology,65,13,20,3,5,3,5,5,8,8,12,8,12,13,20
6,0623,Consumer Discretionary,52,10,19,3,6,2,4,2,4,2,4,3,6,10,19
7,0623,Consumer Staples,38,6,16,7,18,7,18,7,18,2,5,2,5,6,16
8,0623,Communication Services,20,2,10,1,5,1,5,2,10,2,10,2,10,2,10
9,0623,Materials,29,3,10,1,3,1,3,3,10,1,3,1,3,3,10



Imputation Summary - Period 0923


,Period,Sector,Number of Stocks,Missing Scope Count,Missing Scope (%),Filled Scope 1 Count,Filled Scope 1 %,Filled Scope 2 Count,Filled Scope 2 %,Filled Scope 3 Count,Filled Scope 3 %,Imputed Scope 1 Count,Imputed Scope 1 %,Imputed Scope 2 Count,Imputed Scope 2 %,Imputed Scope 3 Count,Imputed Scope 3 %
0,0923,Energy,23,11,48,2,9,2,9,2,9,2,9,2,9,11,48
1,0923,Health Care,64,20,31,2,3,2,3,4,6,7,11,7,11,20,31
2,0923,Industrials,75,22,29,3,4,3,4,3,4,5,7,5,7,22,29
3,0923,Real Estate,31,8,26,0,0,0,0,0,0,4,13,4,13,8,26
4,0923,Financials,72,17,24,2,3,2,3,3,4,7,10,7,10,17,24
5,0923,Information Technology,65,13,20,9,14,9,14,10,15,8,12,8,12,13,20
6,0923,Consumer Discretionary,52,10,19,3,6,3,6,3,6,3,6,3,6,10,19
7,0923,Consumer Staples,37,5,14,9,24,9,24,9,24,2,5,2,5,5,14
8,0923,Communication Services,20,2,10,2,10,2,10,3,15,2,10,2,10,2,10
9,0923,Materials,29,3,10,1,3,1,3,3,10,1,3,1,3,3,10



Imputation Summary - Period 1223


,Period,Sector,Number of Stocks,Missing Scope Count,Missing Scope (%),Filled Scope 1 Count,Filled Scope 1 %,Filled Scope 2 Count,Filled Scope 2 %,Filled Scope 3 Count,Filled Scope 3 %,Imputed Scope 1 Count,Imputed Scope 1 %,Imputed Scope 2 Count,Imputed Scope 2 %,Imputed Scope 3 Count,Imputed Scope 3 %
0,1223,Energy,23,11,48,2,9,2,9,2,9,2,9,2,9,11,48
1,1223,Industrials,77,25,32,4,5,4,5,5,6,8,10,8,10,25,32
2,1223,Health Care,63,19,30,3,5,3,5,4,6,6,10,6,10,19,30
3,1223,Real Estate,31,8,26,0,0,0,0,0,0,4,13,4,13,8,26
4,1223,Financials,72,17,24,4,6,4,6,4,6,7,10,7,10,17,24
5,1223,Consumer Discretionary,53,11,21,6,11,6,11,5,9,4,8,4,8,11,21
6,1223,Information Technology,64,13,20,15,23,15,23,16,25,8,12,8,12,13,20
7,1223,Consumer Staples,37,5,14,12,32,12,32,11,30,2,5,2,5,5,14
8,1223,Communication Services,19,2,11,2,11,2,11,2,11,2,11,2,11,2,11
9,1223,Materials,28,3,11,3,11,3,11,4,14,1,4,1,4,3,11


### Aggregate Imputation Summary (All Periods)

In [12]:
# Combine all imputation summaries
all_imputation = pd.concat(imputation_summaries, ignore_index=True)

# Build aggregation dict dynamically based on available columns
agg_dict = {
    'Number of Stocks': 'mean',
    'Imputed Scope 1 %': 'mean',
    'Imputed Scope 2 %': 'mean',
    'Imputed Scope 3 %': 'mean'
}

# Add filled percentages if they exist in the data
if 'Filled Scope 1 %' in all_imputation.columns:
    agg_dict['Filled Scope 1 %'] = 'mean'
if 'Filled Scope 2 %' in all_imputation.columns:
    agg_dict['Filled Scope 2 %'] = 'mean'
if 'Filled Scope 3 %' in all_imputation.columns:
    agg_dict['Filled Scope 3 %'] = 'mean'

# Aggregate by sector (averaging across periods)
agg_imputation = all_imputation.groupby('Sector').agg(agg_dict).reset_index()

# Round number of stocks to integer
agg_imputation['Number of Stocks'] = agg_imputation['Number of Stocks'].round(0).astype(int)

# Round percentages to integers
percentage_cols = [col for col in agg_imputation.columns if '%' in col]
agg_imputation[percentage_cols] = agg_imputation[percentage_cols].round(0).astype('Int64')
# Round and clean table
percentage_cols = [col for col in agg_imputation.columns if '%' in col]
agg_imputation[percentage_cols] = agg_imputation[percentage_cols].round(0).astype('Int64')

# Export LaTeX table body only
latex_body = agg_imputation.to_latex(
    index=False,
    escape=False,
    header=False,  # We'll handle headers manually in LaTeX
    float_format="%.0f"
)

with open("results/data_summaries/imputation_summary_body.tex", "w") as f:
    f.write(latex_body)

print("✓ Table body (no header) saved as results/imputation_summary_body.tex")



✓ Table body (no header) saved as results/imputation_summary_body.tex


In [13]:
# Combine all imputation summaries
all_imputation = pd.concat(imputation_summaries, ignore_index=True)

# Build aggregation dict dynamically
agg_dict = {
    'Number of Stocks': ['mean', 'std'],
    'Imputed Scope 1 %': ['mean', 'std'],
    'Imputed Scope 2 %': ['mean', 'std'],
    'Imputed Scope 3 %': ['mean', 'std'],
}

# Add filled percentages if present
for col in ['Filled Scope 1 %', 'Filled Scope 2 %', 'Filled Scope 3 %']:
    if col in all_imputation.columns:
        agg_dict[col] = ['mean', 'std']

# Aggregate by sector
agg = all_imputation.groupby('Sector').agg(agg_dict)

agg.columns = ['_'.join(col).strip() for col in agg.columns]
agg = agg.reset_index()

def mean_sd_format(df, base_col):
    mean = df[f"{base_col}_mean"]
    std  = df[f"{base_col}_std"]
    return mean.round(0).astype(int).astype(str) + " (" + std.round(0).astype(int).astype(str) + ")"

formatted = pd.DataFrame()
formatted['Sector'] = agg['Sector']

# Number of stocks (still report mean only, integer)
formatted['Avg. Number of Stocks'] = agg['Number of Stocks_mean'].round(0).astype(int)

# Percentage columns
percentage_bases = [
    'Imputed Scope 1 %',
    'Imputed Scope 2 %',
    'Imputed Scope 3 %',
    'Filled Scope 1 %',
    'Filled Scope 2 %',
    'Filled Scope 3 %'
]

for col in percentage_bases:
    if f"{col}_mean" in agg.columns:
        formatted[col] = mean_sd_format(agg, col)

latex_body = formatted.to_latex(
    index=False,
    escape=False,
    header=False,  # headers handled in LaTeX
)

with open("results/data_summaries/imputation_summary_body.tex", "w") as f:
    f.write(latex_body)

print("✓ Table body with mean (SD) saved")


✓ Table body with mean (SD) saved


## 2. Sector Emissions Summary

### Individual Period Summaries

In [14]:
# Create emissions summary for each period
emissions_summaries = []

for period, df in all_periods_data.items():
    summary = sector_aggregates(df, period)
    emissions_summaries.append(summary)
    
    # Format for display
    summary_display = summary.copy()
    
    pct_cols = ["% Scope 1", "% Scope 2", "% Scope 3", "% Top 5 emitters", 
                "% Emissions in Top 20% Weight", "Sector Market Cap %"]
    summary_display[pct_cols] = summary_display[pct_cols].map(lambda x: f"{x:.1%}")
    
    round_cols = ["Avg Scope 1+2+3", "Avg Scope 1", "Avg Scope 2", "Avg Scope 3",
                  "Var Scope 1", "Var Scope 2", "Var Scope 3",
                  "Avg Carbon Intensity", "Aggregate Carbon Intensity"]
    summary_display[round_cols] = summary_display[round_cols].round(2)
    
    num_cols = ["Scope 1 sum","Scope 2 sum","Scope 3 sum","Scope 1+2+3 absolute"]
    summary_display[num_cols] = summary_display[num_cols].map(human_format)
    
    print(f"\n{'='*80}")
    print(f"Emissions Summary - Period {period}")
    print(f"{'='*80}")
    display(summary_display)


Emissions Summary - Period 0321


,Period,Sector,Scope 1 sum,Scope 2 sum,Scope 3 sum,Scope 1+2+3 absolute,% Scope 1,% Scope 2,% Scope 3,% Top 5 emitters,% Emissions in Top 20% Weight,Avg Scope 1+2+3,Avg Scope 1,Avg Scope 2,Avg Scope 3,Var Scope 1,Var Scope 2,Var Scope 3,Avg Carbon Intensity,Aggregate Carbon Intensity,Sector Market Cap %
0,0321,Communication Services,3.9M,28.3M,91.8M,124.0M,3.1%,22.8%,74.1%,63.8%,0.0%,"5,636,141.64","176,227.40","1,285,993.33","4,173,920.91","285,230.45","1,848,571.31","5,346,029.46",0.23,0.11,10.9%
1,0321,Consumer Discretionary,36.2M,28.0M,1.4B,1.5B,2.4%,1.9%,95.7%,66.1%,0.0%,"25,857,241.19","624,207.66","482,318.97","24,750,714.55","1,939,818.69","900,539.91","61,288,543.95",1.23,0.88,12.3%
2,0321,Consumer Staples,46.2M,36.1M,1.3B,1.3B,3.4%,2.7%,93.9%,59.4%,14.5%,"37,347,456.53","1,282,626.35","1,002,379.00","35,062,451.18","2,556,044.10","1,684,409.29","57,503,495.89",0.79,0.71,6.8%
3,0321,Energy,342.5M,42.8M,3.0B,3.4B,10.1%,1.3%,88.6%,55.9%,0.0%,"147,107,026.96","14,892,685.57","1,858,785.74","130,355,555.65","22,636,333.69","2,240,399.62","127,710,531.52",4.34,3.19,2.9%
4,0321,Financials,7.3M,10.1M,1.6B,1.6B,0.4%,0.6%,98.9%,95.0%,0.1%,"21,921,487.37","98,169.35","136,609.50","21,686,708.52","720,230.59","555,228.16","120,048,492.58",1.95,0.87,14.7%
5,0321,Health Care,10.3M,20.7M,489.6M,520.5M,2.0%,4.0%,94.0%,65.9%,6.0%,"8,262,695.76","163,246.83","328,791.14","7,770,657.79","285,108.89","841,379.91","30,750,810.02",0.48,0.05,12.5%
6,0321,Industrials,205.8M,17.5M,3.0B,3.2B,6.4%,0.5%,93.1%,63.5%,1.1%,"43,478,127.74","2,781,429.12","236,911.21","40,459,787.42","6,265,840.54","298,316.24","144,028,209.05",3.98,2.38,10.2%
7,0321,Information Technology,16.1M,31.5M,289.1M,336.6M,4.8%,9.4%,85.9%,36.1%,0.0%,"5,343,493.88","255,405.31","499,657.68","4,588,430.88","873,077.20","1,086,855.32","6,714,044.90",0.48,0.10,22.2%
8,0321,Materials,144.4M,70.1M,461.3M,675.8M,21.4%,10.4%,68.3%,60.7%,8.7%,"25,030,217.19","5,348,298.74","2,596,797.15","17,085,121.30","7,139,269.01","4,486,483.25","24,822,637.42",1.58,1.58,2.5%
9,0321,Real Estate,2.1M,7.8M,64.0M,73.9M,2.8%,10.6%,86.6%,76.7%,6.5%,"2,547,666.42","71,648.45","270,048.55","2,205,969.42","133,815.86","547,141.11","6,281,876.61",0.33,0.57,2.4%



Emissions Summary - Period 0621


,Period,Sector,Scope 1 sum,Scope 2 sum,Scope 3 sum,Scope 1+2+3 absolute,% Scope 1,% Scope 2,% Scope 3,% Top 5 emitters,% Emissions in Top 20% Weight,Avg Scope 1+2+3,Avg Scope 1,Avg Scope 2,Avg Scope 3,Var Scope 1,Var Scope 2,Var Scope 3,Avg Carbon Intensity,Aggregate Carbon Intensity,Sector Market Cap %
0,0621,Communication Services,3.9M,27.7M,270.6M,302.2M,1.3%,9.2%,89.5%,83.8%,0.0%,"13,737,274.01","179,223.31","1,257,960.42","12,300,090.28","284,183.20","1,858,643.14","40,126,362.10",1.81,0.26,10.9%
1,0621,Consumer Discretionary,36.2M,27.8M,1.4B,1.5B,2.5%,1.9%,95.6%,67.8%,0.0%,"25,057,208.36","624,105.21","478,804.09","23,954,299.05","1,939,851.03","901,961.26","61,205,564.89",1.20,0.85,11.8%
2,0621,Consumer Staples,46.2M,36.1M,1.3B,1.3B,3.4%,2.7%,93.9%,59.6%,14.6%,"37,232,700.00","1,282,626.35","1,002,379.00","34,947,694.65","2,556,044.10","1,684,409.29","57,471,624.20",0.81,0.71,6.6%
3,0621,Energy,337.8M,42.2M,2.9B,3.3B,10.4%,1.3%,88.3%,56.9%,0.0%,"147,985,217.91","15,354,489.45","1,917,548.73","130,713,179.73","23,057,865.38","2,274,907.92","125,858,737.72",4.46,3.13,2.9%
4,0621,Financials,16.9M,9.4M,1.7B,1.7B,1.0%,0.5%,98.5%,96.6%,0.1%,"23,625,008.30","228,506.53","126,622.06","23,269,879.71","1,868,764.63","426,393.90","121,340,113.93",1.81,0.93,14.7%
5,0621,Health Care,16.0M,11.5M,248.1M,275.6M,5.8%,4.2%,90.0%,39.9%,11.4%,"4,374,812.64","254,712.11","181,760.14","3,938,340.39","1,040,601.16","226,267.46","6,317,568.05",0.17,0.03,12.6%
6,0621,Industrials,204.6M,20.5M,3.4B,3.6B,5.6%,0.6%,93.8%,63.8%,2.2%,"48,995,076.05","2,765,349.38","276,485.52","45,953,241.15","6,298,315.66","331,867.79","148,968,795.36",4.77,2.66,9.9%
7,0621,Information Technology,12.7M,38.9M,341.0M,392.6M,3.2%,9.9%,86.9%,40.7%,0.0%,"6,231,468.93","201,192.98","617,860.56","5,412,415.40","555,973.44","1,308,306.47","9,320,253.09",0.63,0.07,23.1%
8,0621,Materials,144.8M,70.7M,453.8M,669.3M,21.6%,10.6%,67.8%,61.2%,8.8%,"23,904,988.86","5,171,573.79","2,525,982.96","16,207,432.11","7,067,949.00","4,418,533.80","24,655,182.56",1.48,1.51,2.6%
9,0621,Real Estate,2.2M,7.8M,64.5M,74.5M,2.9%,10.5%,86.6%,76.1%,6.4%,"2,567,781.86","75,126.55","269,190.21","2,223,465.10","134,446.67","547,543.59","6,280,067.09",0.36,0.58,2.5%



Emissions Summary - Period 0921


...

,Period,Sector,Scope 1 sum,Scope 2 sum,Scope 3 sum,Scope 1+2+3 absolute,% Scope 1,% Scope 2,% Scope 3,% Top 5 emitters,% Emissions in Top 20% Weight,Avg Scope 1+2+3,Avg Scope 1,Avg Scope 2,Avg Scope 3,Var Scope 1,Var Scope 2,Var Scope 3,Avg Carbon Intensity,Aggregate Carbon Intensity,Sector Market Cap %
0,0623,Communication Services,3.2M,32.2M,88.0M,123.4M,2.6%,26.1%,71.3%,68.0%,0.0%,"6,171,325.77","160,312.25","1,612,446.02","4,398,567.50","266,668.47","2,426,725.30","5,138,947.16",0.10,0.01,8.5%
1,0623,Consumer Discretionary,43.5M,25.9M,1.6B,1.6B,2.7%,1.6%,95.8%,68.4%,0.0%,"31,520,848.16","836,108.06","497,908.98","30,186,831.13","2,445,212.58","915,815.29","75,507,247.28",0.77,0.82,11.1%
2,0623,Consumer Staples,55.0M,36.8M,1.3B,1.4B,4.0%,2.7%,93.3%,62.9%,12.9%,"36,091,910.34","1,447,319.81","969,444.91","33,675,145.62","2,701,954.71","1,670,686.38","60,129,120.62",0.68,0.62,6.9%
3,0623,Energy,334.7M,48.7M,3.7B,4.1B,8.2%,1.2%,90.6%,55.5%,0.0%,"178,146,111.91","14,553,752.76","2,119,518.37","161,472,840.78","21,464,920.12","2,512,067.60","163,457,892.47",5.89,3.15,4.3%
4,0623,Financials,5.7M,14.8M,1.3B,1.3B,0.4%,1.2%,98.4%,91.5%,16.1%,"17,783,923.39","79,384.84","205,689.53","17,498,849.02","499,065.23","946,493.99","78,739,391.27",2.33,0.51,12.0%
5,0623,Health Care,7.5M,11.4M,380.8M,399.7M,1.9%,2.9%,95.3%,52.1%,6.6%,"6,245,049.05","117,098.23","178,523.69","5,949,427.13","169,958.03","224,049.75","11,705,352.92",0.54,0.13,13.2%
6,0623,Industrials,239.0M,17.5M,3.6B,3.9B,6.2%,0.5%,93.3%,60.9%,2.3%,"51,346,288.44","3,187,099.14","233,574.26","47,925,615.04","8,168,147.36","271,651.19","149,930,571.24",3.75,2.39,8.7%
7,0623,Information Technology,8.4M,31.1M,263.3M,302.8M,2.8%,10.3%,87.0%,38.4%,0.0%,"4,658,618.42","129,244.81","477,840.02","4,051,533.58","381,112.05","1,206,946.65","6,249,022.19",0.26,0.19,28.0%
8,0623,Materials,141.9M,70.8M,525.7M,738.3M,19.2%,9.6%,71.2%,57.1%,0.0%,"25,457,812.52","4,891,690.13","2,440,051.69","18,126,070.69","6,827,966.18","4,229,973.44","23,897,424.73",1.63,1.56,2.5%
9,0623,Real Estate,3.6M,9.8M,99.1M,112.5M,3.2%,8.7%,88.1%,75.2%,4.8%,"3,629,655.71","116,423.14","316,795.77","3,196,436.79","223,220.20","744,565.75","6,996,492.77",0.77,0.71,2.3%



Emissions Summary - Period 0923


,Period,Sector,Scope 1 sum,Scope 2 sum,Scope 3 sum,Scope 1+2+3 absolute,% Scope 1,% Scope 2,% Scope 3,% Top 5 emitters,% Emissions in Top 20% Weight,Avg Scope 1+2+3,Avg Scope 1,Avg Scope 2,Avg Scope 3,Var Scope 1,Var Scope 2,Var Scope 3,Avg Carbon Intensity,Aggregate Carbon Intensity,Sector Market Cap %
0,0923,Communication Services,3.3M,30.5M,92.4M,126.3M,2.6%,24.2%,73.2%,66.4%,0.0%,"6,313,926.60","165,303.75","1,526,253.85","4,622,369.00","264,910.74","2,449,408.10","5,085,153.02",0.14,0.01,9.3%
1,0923,Consumer Discretionary,44.3M,26.3M,1.6B,1.7B,2.6%,1.5%,95.9%,65.6%,0.0%,"32,842,678.16","851,283.98","506,335.02","31,485,059.16","2,442,164.20","916,146.88","75,454,287.64",1.04,0.85,11.1%
2,0923,Consumer Staples,45.6M,37.0M,1.4B,1.5B,3.1%,2.5%,94.4%,61.5%,12.0%,"39,746,060.27","1,231,082.67","998,803.21","37,516,174.39","2,413,598.33","1,689,268.95","62,110,309.26",0.86,0.67,6.5%
3,0923,Energy,341.7M,49.7M,3.1B,3.5B,9.7%,1.4%,88.8%,53.8%,0.0%,"152,474,868.47","14,855,661.59","2,160,770.54","135,458,436.35","21,291,748.69","2,490,180.44","132,288,251.48",4.52,2.70,4.7%
4,0923,Financials,8.7M,10.3M,1.6B,1.6B,0.5%,0.6%,98.8%,91.2%,8.4%,"22,399,126.29","121,434.73","142,630.13","22,135,061.44","888,997.71","519,630.23","83,297,057.05",3.07,0.64,12.4%
5,0923,Health Care,12.8M,10.8M,555.7M,579.4M,2.2%,1.9%,95.9%,63.3%,4.2%,"9,052,669.34","200,628.85","168,574.88","8,683,465.61","526,098.77","215,453.62","28,254,913.31",0.37,0.19,13.2%
6,0923,Industrials,234.3M,18.2M,3.7B,4.0B,5.9%,0.5%,93.7%,58.6%,1.7%,"53,304,140.63","3,123,677.19","242,954.90","49,937,508.54","8,166,718.15","298,693.85","149,715,993.40",4.13,2.47,8.2%
7,0923,Information Technology,12.3M,36.3M,289.8M,338.4M,3.6%,10.7%,85.6%,34.3%,0.0%,"5,205,863.83","189,471.98","558,454.90","4,457,936.95","432,052.93","1,256,538.29","6,352,948.01",0.43,0.21,28.0%
8,0923,Materials,141.9M,68.1M,463.2M,673.2M,21.1%,10.1%,68.8%,60.0%,0.0%,"23,213,286.10","4,892,669.44","2,348,120.66","15,972,496.00","6,827,344.03","4,248,773.99","23,144,483.84",1.46,1.43,2.4%
9,0923,Real Estate,2.3M,10.4M,79.5M,92.1M,2.4%,11.3%,86.3%,60.8%,3.8%,"2,971,258.65","72,673.53","335,168.71","2,563,416.41","124,213.42","750,221.37","5,685,389.26",0.51,0.58,2.1%



Emissions Summary - Period 1223


,Period,Sector,Scope 1 sum,Scope 2 sum,Scope 3 sum,Scope 1+2+3 absolute,% Scope 1,% Scope 2,% Scope 3,% Top 5 emitters,% Emissions in Top 20% Weight,Avg Scope 1+2+3,Avg Scope 1,Avg Scope 2,Avg Scope 3,Var Scope 1,Var Scope 2,Var Scope 3,Avg Carbon Intensity,Aggregate Carbon Intensity,Sector Market Cap %
0,1223,Communication Services,3.2M,30.8M,139.9M,173.9M,1.9%,17.7%,80.4%,74.7%,0.0%,"9,151,145.58","169,337.32","1,620,843.52","7,360,964.74","270,934.39","2,488,415.95","13,151,817.21",0.21,0.14,8.9%
1,1223,Consumer Discretionary,45.0M,26.1M,1.6B,1.6B,2.8%,1.6%,95.6%,68.8%,0.0%,"30,720,877.32","849,167.53","491,898.71","29,379,811.09","2,419,544.74","907,690.82","74,641,943.83",0.81,0.80,11.2%
2,1223,Consumer Staples,60.1M,36.0M,1.3B,1.4B,4.2%,2.5%,93.3%,60.1%,12.3%,"38,924,049.21","1,624,280.91","972,897.37","36,326,870.93","3,257,419.06","1,692,000.60","60,830,594.09",0.69,0.65,6.1%
3,1223,Energy,335.4M,50.0M,3.2B,3.6B,9.3%,1.4%,89.3%,55.8%,0.0%,"156,993,237.01","14,580,497.24","2,172,591.80","140,240,147.97","21,446,435.23","2,498,578.30","137,671,926.81",4.50,2.78,3.8%
4,1223,Financials,2.9M,10.2M,1.1B,1.1B,0.3%,0.9%,98.8%,92.0%,4.8%,"15,297,631.10","40,941.51","141,164.31","15,115,525.28","144,699.57","520,345.69","76,011,736.69",2.26,0.43,12.6%
5,1223,Health Care,8.2M,11.9M,475.6M,495.8M,1.7%,2.4%,95.9%,62.4%,5.0%,"7,869,356.95","130,761.79","189,151.18","7,549,443.99","173,689.31","238,058.37","24,885,972.08",0.32,0.16,12.6%
6,1223,Industrials,231.6M,18.1M,3.7B,3.9B,5.9%,0.5%,93.7%,59.6%,1.5%,"51,065,573.98","3,007,151.68","234,983.55","47,823,438.75","8,050,167.53","284,334.99","148,790,079.28",3.86,2.38,8.6%
7,1223,Information Technology,13.1M,32.4M,281.5M,327.1M,4.0%,9.9%,86.1%,35.5%,0.0%,"5,110,252.43","204,400.43","506,704.16","4,399,147.83","629,999.76","1,232,178.78","6,410,069.21",0.47,0.20,29.5%
8,1223,Materials,141.4M,68.1M,710.1M,919.6M,15.4%,7.4%,77.2%,66.5%,0.0%,"32,844,106.75","5,050,797.06","2,431,073.54","25,362,236.14","6,897,872.14","4,303,192.58","48,505,747.33",1.94,0.04,2.4%
9,1223,Real Estate,3.4M,9.8M,73.9M,87.1M,3.9%,11.3%,84.9%,66.4%,6.2%,"2,808,444.78","108,153.30","316,570.68","2,383,720.79","189,812.37","745,522.81","5,715,593.76",0.45,0.55,2.2%


### Aggregate Emissions Summary (All Periods)

In [32]:
# Combine all emissions summaries
all_emissions = pd.concat(emissions_summaries, ignore_index=True)

# Aggregate by sector (averaging across periods)
agg_emissions = all_emissions.groupby('Sector').agg({
    'Scope 1 sum': 'mean',
    'Scope 2 sum': 'mean',
    'Scope 3 sum': 'mean',
    'Scope 1+2+3 absolute': 'mean',
    '% Scope 1': 'mean',
    '% Scope 2': 'mean',
    '% Scope 3': 'mean',
    '% Top 5 emitters': 'mean',
    '% Emissions in Top 20% Weight': 'mean',
    'Avg Scope 1+2+3': 'mean',
    'Avg Scope 1': 'mean',
    'Avg Scope 2': 'mean',
    'Avg Scope 3': 'mean',
    'Var Scope 1': 'mean',
    'Var Scope 2': 'mean',
    'Var Scope 3': 'mean',
    'Avg Carbon Intensity': 'mean',
    'Aggregate Carbon Intensity': 'mean',
    'Sector Market Cap %': 'mean'
}).reset_index()

# Load sector volatility data and merge with emissions summary
volatility_file = Path("data/benchmark_returns_volatility/sector_annualized_volatility_2021_2023.xlsx")

if volatility_file.exists():
    volatility_df = pd.read_excel(volatility_file, index_col=0)
    
    # The volatility file has sector names as index and a single column with volatility values
    # Convert to a DataFrame with 'Sector' and 'Annualized Volatility' columns
    volatility_df = volatility_df.reset_index()
    volatility_df.columns = ['Sector', 'Annualized Volatility']
    
    # Merge with aggregate emissions
    agg_emissions = agg_emissions.merge(volatility_df, on='Sector', how='left')
    
    print(f"\n✓ Successfully merged volatility data")
    print(f"  Added 'Annualized Volatility' column to aggregate emissions summary")
else:
    print(f"\n⚠ Volatility file not found at: {volatility_file}")
    print("  Run 'benchmark_replication.py' first to generate volatility data")

# Sort by total emissions
agg_emissions = agg_emissions.sort_values('Scope 1+2+3 absolute', ascending=False).reset_index(drop=True)

# Format for display
agg_emissions_display = agg_emissions.copy()

pct_cols = ["% Scope 1", "% Scope 2", "% Scope 3", "% Top 5 emitters", 
            "% Emissions in Top 20% Weight", "Sector Market Cap %"]
agg_emissions_display[pct_cols] = agg_emissions_display[pct_cols].map(lambda x: f"{x:.1%}")

round_cols = ["Avg Scope 1+2+3", "Avg Scope 1", "Avg Scope 2", "Avg Scope 3",
              "Var Scope 1", "Var Scope 2", "Var Scope 3",
              "Avg Carbon Intensity", "Aggregate Carbon Intensity"]
agg_emissions_display[round_cols] = agg_emissions_display[round_cols].round(2)

# Format Annualized Volatility if it exists
if 'Annualized Volatility' in agg_emissions_display.columns:
    agg_emissions_display['Annualized Volatility'] = agg_emissions_display['Annualized Volatility'].map(lambda x: f"{x:.2%}")

num_cols = ["Scope 1 sum","Scope 2 sum","Scope 3 sum","Scope 1+2+3 absolute"]
agg_emissions_display[num_cols] = agg_emissions_display[num_cols].map(human_format)

print(f"\n{'='*80}")
print(f"AGGREGATE EMISSIONS SUMMARY (Average across {len(PERIODS)} periods)")
print(f"{'='*80}")
display(agg_emissions_display)


✓ Successfully merged volatility data
  Added 'Annualized Volatility' column to aggregate emissions summary

AGGREGATE EMISSIONS SUMMARY (Average across 12 periods)


,Sector,Scope 1 sum,Scope 2 sum,Scope 3 sum,Scope 1+2+3 absolute,% Scope 1,% Scope 2,% Scope 3,% Top 5 emitters,% Emissions in Top 20% Weight,Avg Scope 1+2+3,Avg Scope 1,Avg Scope 2,Avg Scope 3,Var Scope 1,Var Scope 2,Var Scope 3,Avg Carbon Intensity,Aggregate Carbon Intensity,Sector Market Cap %,Annualized Volatility
0,Industrials,223.2M,18.8M,3.6B,3.9B,5.8%,0.5%,93.7%,59.6%,2.1%,"51,618,553.82","2,975,609.63","251,086.95","48,391,857.24","7,236,528.27","322,227.19","147,939,596.83",4.45,1.91,8.8%,17.25%
1,Energy,336.3M,45.5M,3.1B,3.5B,9.7%,1.3%,88.9%,54.7%,0.0%,"157,368,363.24","15,250,319.63","2,061,394.41","140,056,649.19","22,325,828.61","2,432,021.23","131,458,539.00",4.60,2.75,3.9%,28.47%
2,Consumer Discretionary,40.9M,26.5M,1.4B,1.5B,2.7%,1.8%,95.5%,67.6%,0.0%,"27,703,876.43","756,278.02","487,646.63","26,459,951.77","2,220,318.81","903,149.54","68,011,121.86",0.94,0.80,11.4%,25.68%
3,Consumer Staples,49.2M,36.4M,1.3B,1.4B,3.5%,2.6%,93.9%,60.6%,13.1%,"38,173,946.89","1,336,949.43","990,273.18","35,846,724.27","2,648,060.96","1,687,801.73","59,793,527.33",0.78,0.68,6.9%,13.61%
4,Utilities,648.9M,26.3M,665.9M,1.3B,48.4%,2.0%,49.7%,36.5%,3.4%,"47,208,576.28","22,850,888.63","928,597.67","23,429,089.98","21,650,713.94","1,814,102.13","19,794,754.99",3.83,3.58,2.6%,18.32%
5,Financials,9.2M,10.4M,1.2B,1.2B,0.9%,0.9%,98.1%,91.9%,10.3%,"16,935,306.37","124,047.15","141,561.45","16,669,697.77","863,503.68","531,244.46","82,273,076.70",1.81,0.55,13.4%,19.37%
6,Materials,144.1M,69.2M,502.7M,716.0M,20.3%,9.7%,70.0%,59.5%,5.9%,"25,355,767.31","5,101,306.00","2,451,265.70","17,803,195.61","6,967,366.83","4,329,886.32","25,951,782.28",1.52,1.36,2.5%,19.54%
7,Health Care,11.4M,12.5M,423.2M,447.0M,2.6%,2.9%,94.4%,51.6%,7.3%,"7,076,893.78","179,797.44","198,094.92","6,699,001.41","467,405.65","316,976.80","17,491,212.70",0.34,0.12,13.7%,14.27%
8,Information Technology,16.1M,33.5M,326.7M,376.2M,4.1%,9.3%,86.6%,40.5%,0.0%,"5,935,175.70","253,855.89","527,752.50","5,153,567.30","1,006,375.16","1,241,717.71","9,889,506.33",0.45,0.13,25.0%,25.21%
9,Communication Services,8.8M,31.0M,177.8M,217.7M,3.6%,17.7%,78.8%,73.0%,0.0%,"9,869,639.77","393,633.19","1,432,397.51","8,043,609.07","1,209,176.01","2,159,948.79","20,481,817.19",0.30,0.09,9.3%,24.08%


In [16]:
volatility_df['Annualized Volatility'].mean()

0.2056809907474607

In [17]:


def compute_sector_structural_metrics(all_periods_data):
    """
    Computes the structural metrics for each sector across periods:
      1a. HHI of benchmark weights
      1b. Largest benchmark weight
      2a. Top 10% carbon intensity weight share
      2b. Weight needed to reach 50% of sector carbon footprint
      2c. Coefficient of variation (CV) of carbon intensity
      4.  Number of constituents

    Returns:
        DataFrame with sectors as rows and metrics averaged across periods.
    """

    results = []

    for period, df in all_periods_data.items():

        # Clean:
        df = df.copy()
        df["Carbon Intensity"] = df["Carbon Intensity"].astype(float)
        df["weight_in_sector"] = df["weight_in_sector"].astype(float)

        # Group by sector
        for sector, sdf in df.groupby("GICS Sector"):

            weights = sdf["weight_in_sector"].values
            carb = sdf["Carbon Intensity"].values

            # -----------------------------
            # 1a. HHI
            # -----------------------------
            hhi = np.sum(weights**2)

            # -----------------------------
            # 1b. Largest constituent weight
            # -----------------------------
            largest_weight = weights.max()

            # -----------------------------
            # 2a. Top 10% most carbon intensive weight share
            # -----------------------------
            n_top = max(1, int(len(sdf) * 0.10))
            sdf_sorted = sdf.sort_values("Carbon Intensity", ascending=False)
            top10_weight = sdf_sorted.head(n_top)["weight_in_sector"].sum()

            # -----------------------------
            # 2b. Weight needed to reach 50% of sector carbon footprint
            # -----------------------------
            sdf_sorted["carbon_contrib"] = sdf_sorted["weight_in_sector"] * sdf_sorted["Carbon Intensity"]
            total_carbon = sdf_sorted["carbon_contrib"].sum()

            sdf_sorted["cum_carbon"] = sdf_sorted["carbon_contrib"].cumsum()
            cutoff_row = sdf_sorted[sdf_sorted["cum_carbon"] >= 0.5 * total_carbon].iloc[0]
            weight_to_half_carbon = sdf_sorted.loc[:cutoff_row.name]["weight_in_sector"].sum()

            # -----------------------------
            # 2c. Coefficient of variation of carbon intensity
            # -----------------------------
            if carb.mean() > 0:
                cv_carbon = carb.std() / carb.mean()
            else:
                cv_carbon = np.nan

            # -----------------------------
            # 4. Number of constituents
            # -----------------------------
            n_const = len(sdf)

            results.append({
                "Period": period,
                "Sector": sector,
                "HHI": hhi,
                "Largest Weight": largest_weight,
                "Top10% CI Weight": top10_weight,
                "Weight to 50% Carbon": weight_to_half_carbon,
                "CV Carbon Intensity": cv_carbon,
                "Num Constituents": n_const
            })

    # Convert to DataFrame
    results_df = pd.DataFrame(results)

    # Average across periods
    aggregated = (
        results_df
        .groupby("Sector")
        .mean(numeric_only=True)
        .reset_index()
        .sort_values("Sector")
    )

    return aggregated


In [18]:
metrics_table = compute_sector_structural_metrics(all_periods_data)
print(metrics_table)

vol_2123 = pd.read_excel(
    "data/benchmark_returns_volatility/sector_annualized_volatility_2021_2023.xlsx",
    header=0
)

# Clean: ensure column names are correct
vol_2123.columns = ["Sector", "annual_vol_21_23"]

# Merge on "Sector"
metrics_table = metrics_table.merge(
    vol_2123,
    on="Sector",
    how="left"
)

print(metrics_table)


                    Sector  HHI  Largest Weight  Top10% CI Weight  \
0   Communication Services 0.23            0.42              0.02   
1   Consumer Discretionary 0.14            0.31              0.06   
2         Consumer Staples 0.07            0.14              0.21   
3                   Energy 0.13            0.26              0.03   
4               Financials 0.04            0.09              0.04   
5              Health Care 0.04            0.10              0.05   
6              Industrials 0.03            0.07              0.05   
7   Information Technology 0.15            0.27              0.03   
8                Materials 0.07            0.19              0.06   
9              Real Estate 0.05            0.12              0.07   
10               Utilities 0.07            0.17              0.04   

    Weight to 50% Carbon  CV Carbon Intensity  Num Constituents  
0                   0.35                 1.84             21.75  
1                   0.13               

In [19]:
metrics_table = metrics_table.rename(columns={
    "HHI": "HHI (Benchmark Concentration)",
    "Largest Weight": "Largest Constituent Weight",
    "Top10% CI Weight": "Weight in Top 10% CI Firms",
    "Weight to 50% Carbon": "Weight to 50% Carbon Contribution",
    "CV Carbon Intensity": "CV Carbon Intensity",
    "Num Constituents": "Number of Constituents",
    "annual_vol_21_23": "Annualized Volatility 2021–2023"
})


In [20]:
pd.merge(avg_hhi, metrics_table, how='left', on = 'Sector')

,Sector,Avg_HHI_optimized,HHI (Benchmark Concentration),Largest Constituent Weight,Weight in Top 10% CI Firms,Weight to 50% Carbon Contribution,CV Carbon Intensity,Number of Constituents,Annualized Volatility 2021–2023
0,Communication Services,22.42,0.23,0.42,0.02,0.35,1.84,21.75,0.24
1,Consumer Discretionary,14.23,0.14,0.31,0.06,0.13,1.32,54.33,0.26
2,Consumer Staples,6.89,0.07,0.14,0.21,0.27,0.86,36.75,0.14
3,Energy,17.03,0.13,0.26,0.03,0.25,0.83,22.08,0.28
4,Financials,5.09,0.04,0.09,0.04,0.01,5.67,73.67,0.19
5,Health Care,5.26,0.04,0.10,0.05,0.16,2.77,63.17,0.14
6,Industrials,4.94,0.03,0.07,0.05,0.05,2.14,75.00,0.17
7,Information Technology,14.15,0.15,0.27,0.03,0.10,2.05,63.50,0.25
8,Materials,7.32,0.07,0.19,0.06,0.38,1.22,28.25,0.20
9,Real Estate,7.30,0.05,0.12,0.07,0.20,1.61,30.42,0.20


In [21]:
dri = pd.read_excel("results/DRI/decarbonization_readiness_index.xlsx")

In [22]:
metrics_table = pd.merge(metrics_table, dri, how= 'left', on = 'Sector')

In [26]:
metrics_table[[ 'HHI (Benchmark Concentration)', 'Sens_norm']].corr()

,HHI (Benchmark Concentration),Sens_norm
HHI (Benchmark Concentration),1.00,0.97
Sens_norm,0.97,1.00


In [31]:
metrics_table.iloc[:, 1:].corr()

,HHI (Benchmark Concentration),Largest Constituent Weight,Weight in Top 10% CI Firms,Weight to 50% Carbon Contribution,CV Carbon Intensity,Number of Constituents,Annualized Volatility 2021–2023,Room_norm,Flex_norm,Sens_norm,Robust_norm,DRI
HHI (Benchmark Concentration),1.00,0.98,-0.31,0.31,-0.28,-0.42,0.73,-0.33,-0.58,0.97,0.40,0.25
Largest Constituent Weight,0.98,1.00,-0.32,0.39,-0.35,-0.48,0.74,-0.39,-0.62,0.96,0.45,0.21
Weight in Top 10% CI Firms,-0.31,-0.32,1.00,0.10,-0.23,-0.08,-0.57,-0.33,-0.02,-0.21,-0.59,-0.60
Weight to 50% Carbon Contribution,0.31,0.39,0.10,1.00,-0.69,-0.90,0.00,-0.89,-0.44,0.27,0.10,-0.54
CV Carbon Intensity,-0.28,-0.35,-0.23,-0.69,1.00,0.68,-0.15,0.77,0.74,-0.22,-0.21,0.59
Number of Constituents,-0.42,-0.48,-0.08,-0.90,0.68,1.00,-0.26,0.89,0.43,-0.37,-0.36,0.36
Annualized Volatility 2021–2023,0.73,0.74,-0.57,0.00,-0.15,-0.26,1.00,-0.01,-0.36,0.70,0.87,0.62
Room_norm,-0.33,-0.39,-0.33,-0.89,0.77,0.89,-0.01,1.00,0.58,-0.27,-0.08,0.68
Flex_norm,-0.58,-0.62,-0.02,-0.44,0.74,0.43,-0.36,0.58,1.00,-0.56,-0.18,0.42
Sens_norm,0.97,0.96,-0.21,0.27,-0.22,-0.37,0.70,-0.27,-0.56,1.00,0.33,0.27


## Save Results

In [33]:
# Create results directory
RESULTS_DIR = Path("results/data_summaries/")
RESULTS_DIR.mkdir(exist_ok=True)

# Save aggregate summaries
output_file = RESULTS_DIR / "sector_summaries_aggregate.xlsx"

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # Save aggregate imputation summary
    agg_imputation.to_excel(writer, sheet_name='Imputation Summary', index=False)
    
    # Save aggregate emissions summary (raw numbers, not formatted)
    agg_emissions.to_excel(writer, sheet_name='Emissions Summary', index=False)
    
    # Save all periods imputation data
    all_imputation.to_excel(writer, sheet_name='All Periods Imputation', index=False)
    
    # Save all periods emissions data
    all_emissions.to_excel(writer, sheet_name='All Periods Emissions', index=False)
    
    # Save filled counts by period
    if 'filled_counts_df' in locals():
        filled_counts_df.to_excel(writer, sheet_name='Filled Counts by Period', index=False)

print(f"\nResults saved to: {output_file}")
print(f"  - Imputation Summary (aggregate)")
print(f"  - Emissions Summary (aggregate)")
print(f"  - All Periods Imputation (detailed)")
print(f"  - All Periods Emissions (detailed)")
if 'filled_counts_df' in locals():
    print(f"  - Filled Counts by Period")


Results saved to: results/data_summaries/sector_summaries_aggregate.xlsx
  - Imputation Summary (aggregate)
  - Emissions Summary (aggregate)
  - All Periods Imputation (detailed)
  - All Periods Emissions (detailed)
